In [0]:
%run "./00_config"

## Standardization Framework — Core Functions

This notebook contains all reusable helper functions used by the Bronze → Silver
standardization engine (`02_standardization_engine`).

**Function groups**
1. Column name standardization (`snake_case`)
2. Metadata-driven mapping lookup (`get_metadata`)
3. Data cleansing (`trim_string_columns`)
4. Audit columns (`add_audit_columns`)
5. Deduplication (`deduplicate`)
6. Data quality checks (`data_quality_check`)
7. Logging (`log`)


## 1. Standardize Column Names (snake_case)

Converts any incoming column naming convention (camelCase, PascalCase, spaces,
special characters, etc.) into a consistent `snake_case` format.

Example: `"CustomerID "` → `"customer_id"`, `"Invoice-Amount"` → `"invoice_amount"`

In [0]:
def snake_case(col: str) -> str:
    """
    Convert a column name to snake_case.

    Steps:
        1. Strip leading/trailing whitespace
        2. Insert underscore between a lowercase/digit and an uppercase letter
        3. Replace any non-alphanumeric character with underscore
        4. Collapse multiple underscores into one
        5. Lowercase and strip leading/trailing underscores

    Args:
        col: original column name

    Returns:
        snake_case version of the column name
    """
    col = col.strip()
    col = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col)
    col = re.sub(r'[^A-Za-z0-9]', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.lower().strip("_")

## 2. Metadata Lookup

Reads the column-mapping / transformation rules for a given source table from the
central metadata table (`METADATA_TABLE`, defined in `00_Config`). Only `active`
rows are returned, ordered by `ordinal_position` so downstream column order is
deterministic.

**Expected metadata columns (required):**
`src_catalog, src_schema, src_table, active, ordinal_position, src_column,
tgt_column, tgt_dtype, default_value, tgt_catalog, tgt_schema, tgt_table`

**Optional columns**:
- `is_primary_key` (bool) — used for deduplication
- `is_mandatory` (bool) — used for data quality null-checks
- `partition_column` (bool) — used to partition the Silver table on write
- `trim_string` (bool) — whether to trim a specific string column (defaults to
  trimming ALL string columns if omitted entirely)

In [0]:
def get_metadata(src_catalog, src_schema, src_table):
    """
    Fetch active column-mapping metadata for a source table.

    Args:
        src_catalog, src_schema, src_table: identifiers of the Bronze source table

    Returns:
        Spark DataFrame of active metadata rows, ordered by ordinal_position
    """
    return (
        spark.table(METADATA_TABLE)
        .filter(
            (F.col("src_catalog") == src_catalog) &
            (F.col("src_schema") == src_schema) &
            (F.col("src_table") == src_table) &
            (F.col("active") == True)
        )
        .orderBy("ordinal_position")
    )

## 3. String Cleansing

Trims leading/trailing whitespace on all string columns and converts empty
strings (`""`) to `NULL` so downstream null-checks behave consistently.
This is a very common Bronze → Silver requirement since raw/source systems
frequently contain inconsistent padding and blank strings instead of nulls.

In [0]:

def trim_string_columns(df):
    """
    Trim whitespace on all string-typed columns and convert blank strings to NULL.

    Args:
        df: input Spark DataFrame

    Returns:
        DataFrame with cleansed string columns (built in a single select — no
        per-column withColumn loop)
    """
    exprs = []
    for field in df.schema.fields:
        c = F.col(field.name)
        if isinstance(field.dataType, StringType):
            trimmed = F.trim(c)
            exprs.append(
                F.when(trimmed == "", None).otherwise(trimmed).alias(field.name)
            )
        else:
            exprs.append(c)
    return df.select(*exprs)

## 4. Audit Columns

Adds standard audit columns to every Silver table so records can always be
traced back to their Bronze source and load batch. This is standard practice
for any governed Bronze → Silver pipeline.

Columns added:
- `_source_catalog`, `_source_schema`, `_source_table` — lineage
- `_ingestion_timestamp` — when the record was processed into Silver
- `_batch_id` — unique id per pipeline run, useful for reprocessing/debugging

In [0]:
def add_audit_columns(df, src_catalog, src_schema, src_table, batch_id=None):
    """
    Attach standard audit columns to a DataFrame.

    Args:
        df: input DataFrame
        src_catalog, src_schema, src_table: Bronze source identifiers
        batch_id: optional pre-generated batch id (e.g. shared across a multi-table
            run). If not provided, a new uuid4 is generated.

    Returns:
        DataFrame with audit columns appended
    """
    batch_id = batch_id or str(uuid.uuid4())
    return (
        df
        .withColumn("_source_catalog", F.lit(src_catalog))
        .withColumn("_source_schema", F.lit(src_schema))
        .withColumn("_source_table", F.lit(src_table))
        .withColumn("_ingestion_timestamp", F.lit(datetime.now()).cast("timestamp"))
        .withColumn("_batch_id", F.lit(batch_id))
    )

## 5. Deduplication

Removes duplicate records. If primary key column(s) are supplied, duplicates are
resolved by keeping the most recently ingested row (via `_ingestion_timestamp`,
so this should run **after** `add_audit_columns`). If no key columns are given,
falls back to a full-row `dropDuplicates()`.

In [0]:
from pyspark.sql import Window

def deduplicate(df, key_columns=None, order_column="_ingestion_timestamp"):
    """
    Deduplicate a DataFrame.

    Args:
        df: input DataFrame (should already contain `order_column` if key_columns given)
        key_columns: list of column names that define a unique record. If None,
            a plain dropDuplicates() over all columns is used instead.
        order_column: column used to decide which duplicate to keep (latest wins)

    Returns:
        Deduplicated DataFrame
    """
    if not key_columns:
        return df.dropDuplicates()

    window = Window.partitionBy(*key_columns).orderBy(F.col(order_column).desc())
    return (
        df.withColumn("_dedup_rank", F.row_number().over(window))
        .filter(F.col("_dedup_rank") == 1)
        .drop("_dedup_rank")
    )

## 6. Logging

Lightweight debug logger controlled by the `DEBUG` flag in `00_Config`.
Includes a timestamp and level so pipeline output is easier to troubleshoot.

In [0]:
def log(msg, level="INFO"):
    """
    Print a timestamped log line when DEBUG is enabled.

    Args:
        msg: message to log
        level: severity label, e.g. INFO / WARN / ERROR
    """
    if DEBUG:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] [{level}] {msg}")